# monoT5 Integrated Gradients

Pointwise Integrated Gradients for `castorini/monot5-base-msmarco`, computed over encoder input embeddings and saved in a pairwise-friendly structure for downstream faithfulness evaluation.


In [1]:
# -- IMPORTS --
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from captum.attr import IntegratedGradients
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


/Users/evelinalune/Documents/uni/MSc-IS/thesis/ig-thesis/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
model_name = "castorini/monot5-base-msmarco"
out_dir = Path("../outputs_monot5")
pair_paths = [
    out_dir / "pairwise_scores.parquet",
    out_dir / "pairwise_scores.pkl",
    out_dir / "pairwise_scores.csv",
]

n_steps = 100
max_length = 512
seed = 42

torch.manual_seed(seed)
np.random.seed(seed)


In [3]:
def load_pairs(paths):
    for path in paths:
        if not path.exists():
            continue
        if path.suffix == ".parquet":
            try:
                return pd.read_parquet(path)
            except ImportError:
                continue
        if path.suffix in {".pkl", ".pickle"}:
            return pd.read_pickle(path)
        if path.suffix == ".csv":
            return pd.read_csv(path)
    raise FileNotFoundError("No readable monoT5 pairwise score file was found.")

pairs_df = load_pairs(pair_paths)
print(f"Loaded {len(pairs_df):,} monoT5 preference pairs")
pairs_df.head()


Loaded 90 monoT5 preference pairs


,qid,query,pid_i,passage_i,score_i,pid_j,passage_j,score_j,g_score,correct_pref
0,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.998757,2442388,"Lebanon, Illinois. Lebanon is a city in St. Cl...",9.834903e-07,0.998756,1
1,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.998757,4012312,"1 On July 1, 1995, became a campus of the Univ...",2.192997e-06,0.998755,1
2,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.998757,2536875,Arctic Climate Research at the University of I...,1.353453e-06,0.998755,1
3,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.998757,4872508,Cost of Attendance. Cost of Attendance (COA) i...,4.307217e-05,0.998714,1
4,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.998757,7759529,Tuition and fees at Keiser University-Ft Laude...,2.301207e-06,0.998754,1


In [4]:
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
except Exception as e:
    raise ImportError(
        "Failed to load the monoT5 tokenizer. Install the required tokenizer dependencies, "
        "for example `pip install sentencepiece protobuf`, restart the kernel, and rerun."
    ) from e

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
model = model.to(device)
model.eval()

embedding_layer = model.get_input_embeddings()
decoder_start_token_id = model.config.decoder_start_token_id
if decoder_start_token_id is None:
    decoder_start_token_id = tokenizer.pad_token_id

true_ids = tokenizer.encode("true", add_special_tokens=False)
false_ids = tokenizer.encode("false", add_special_tokens=False)
if len(true_ids) != 1 or len(false_ids) != 1:
    raise ValueError("Expected 'true' and 'false' to map to single tokens for monoT5 scoring.")

true_token_id = int(true_ids[0])
false_token_id = int(false_ids[0])

print(f"Loaded: {model_name}")
print(f"Device: {device}")
print(f"decoder_start_token_id: {decoder_start_token_id}")
print(f"true token id: {true_token_id}")
print(f"false token id: {false_token_id}")


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Loaded: castorini/monot5-base-msmarco
Device: mps
decoder_start_token_id: 0
true token id: 1176
false token id: 6136


In [5]:
def mono_input(query, passage):
    return f"Query: {query} Document: {passage} Relevant:"

def ids_to_embeds(input_ids):
    return embedding_layer(input_ids)

def _tok_len(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def tokenize_mono(query, passage, max_length=max_length):
    text = mono_input(query, passage)
    encoded = tokenizer(
        text,
        max_length=max_length,
        truncation=True,
        padding=False,
        return_tensors="pt",
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())

    query_prefix = "Query: "
    doc_prefix = " Document: "
    suffix = " Relevant:"

    c1 = query_prefix
    c2 = c1 + query
    c3 = c2 + doc_prefix
    c4 = c3 + passage
    c5 = c4 + suffix

    l1 = _tok_len(c1)
    l2 = _tok_len(c2)
    l3 = _tok_len(c3)
    l4 = _tok_len(c4)
    l5 = _tok_len(c5)

    expected_full_len = l5 + 1
    if input_ids.shape[1] != expected_full_len:
        print(
            f"Warning: segment length mismatch. Expected {expected_full_len}, got {input_ids.shape[1]}. "
            "This can happen due to truncation; positions will be clipped."
        )

    query_positions = [pos for pos in range(l1, min(l2, input_ids.shape[1]))]
    doc_positions = [pos for pos in range(l3, min(l4, input_ids.shape[1]))]

    return {
        "text": text,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "tokens": tokens,
        "query_positions": query_positions,
        "doc_positions": doc_positions,
    }


In [6]:
def forward_mono_true_prob_from_embeds(input_embeds, attention_mask):
    batch_size = input_embeds.shape[0]
    decoder_input_ids = torch.full(
        (batch_size, 1),
        decoder_start_token_id,
        dtype=torch.long,
        device=input_embeds.device,
    )

    outputs = model(
        inputs_embeds=input_embeds,
        attention_mask=attention_mask,
        decoder_input_ids=decoder_input_ids,
    )

    logits = outputs.logits[:, 0, :]
    tf_logits = logits[:, [false_token_id, true_token_id]]
    true_prob = torch.softmax(tf_logits, dim=-1)[:, 1]
    return true_prob

def predict_mono_point(query, passage):
    tok = tokenize_mono(query, passage)
    decoder_input_ids = torch.full(
        (1, 1),
        decoder_start_token_id,
        dtype=torch.long,
        device=device,
    )

    with torch.no_grad():
        outputs = model(
            input_ids=tok["input_ids"],
            attention_mask=tok["attention_mask"],
            decoder_input_ids=decoder_input_ids,
        )

    logits = outputs.logits[:, 0, :]
    tf_logits = logits[:, [false_token_id, true_token_id]]
    true_prob = torch.softmax(tf_logits, dim=-1)[:, 1].item()
    margin = (logits[:, true_token_id] - logits[:, false_token_id]).item()
    return true_prob, margin

def make_baseline_input_ids(input_ids):
    baseline_ids = torch.full_like(input_ids, tokenizer.pad_token_id)
    eos_id = tokenizer.eos_token_id
    if eos_id is not None:
        for pos, token_id in enumerate(input_ids[0].tolist()):
            if token_id == eos_id:
                baseline_ids[0, pos] = eos_id
    return baseline_ids

def make_baseline_embeds(input_ids):
    baseline_ids = make_baseline_input_ids(input_ids)
    return ids_to_embeds(baseline_ids).detach()


In [7]:
def merge_sentencepiece(tokens, scores):
    special_tokens = set(tokenizer.all_special_tokens)
    word_tokens, word_scores = [], []
    current_word, current_score = "", 0.0

    for token, score in zip(tokens, scores):
        if token in special_tokens:
            if current_word:
                word_tokens.append(current_word)
                word_scores.append(current_score)
                current_word, current_score = "", 0.0
            continue

        if token.startswith("▁"):
            if current_word:
                word_tokens.append(current_word)
                word_scores.append(current_score)
            current_word = token.lstrip("▁") or token
            current_score = score
        else:
            current_word += token
            current_score += score

    if current_word:
        word_tokens.append(current_word)
        word_scores.append(current_score)

    return word_tokens, np.array(word_scores)

def aggregate_span(tokens, token_scores, positions):
    span_tokens = [tokens[pos] for pos in positions if pos < len(tokens)]
    span_scores = np.array([token_scores[pos] for pos in positions if pos < len(token_scores)])
    word_tokens, word_scores = merge_sentencepiece(span_tokens, span_scores)
    return {
        "tokens": span_tokens,
        "token_scores": span_scores,
        "word_tokens": word_tokens,
        "word_scores": word_scores,
        "positions": positions,
    }

def aggregate_attributions(attributions, tok):
    token_scores = attributions[0].sum(dim=-1).detach().cpu().numpy()
    full_word_tokens, full_word_scores = merge_sentencepiece(tok["tokens"], token_scores)
    return {
        "tokens": tok["tokens"],
        "token_scores": token_scores,
        "word_tokens": full_word_tokens,
        "word_scores": full_word_scores,
        "query": aggregate_span(tok["tokens"], token_scores, tok["query_positions"]),
        "doc": aggregate_span(tok["tokens"], token_scores, tok["doc_positions"]),
    }


In [8]:
def compute_pointwise_ig_monot5(query, passage):
    tok = tokenize_mono(query, passage)
    input_embeds = ids_to_embeds(tok["input_ids"]).detach()
    baseline_embeds = make_baseline_embeds(tok["input_ids"])

    ig = IntegratedGradients(forward_mono_true_prob_from_embeds)
    attributions, delta = ig.attribute(
        inputs=input_embeds,
        baselines=baseline_embeds,
        additional_forward_args=(tok["attention_mask"],),
        n_steps=n_steps,
        return_convergence_delta=True,
    )

    true_prob, margin = predict_mono_point(query, passage)

    return {
        "method": "pointwise_ig_monot5",
        "input_text": tok["text"],
        "true_prob": float(true_prob),
        "margin": float(margin),
        **aggregate_attributions(attributions, tok),
        "convergence_delta": float(delta.detach().cpu().item()) if torch.is_tensor(delta) else float(delta),
    }


In [9]:
test_row = pairs_df[pairs_df["correct_pref"] == 1].iloc[0]

print(f"Query: {test_row['query']}")
print(f"Passage i: {test_row['passage_i'][:120]}...")
print(f"Passage j: {test_row['passage_j'][:120]}...")
print(f"Stored g score (prob difference): {test_row['g_score']:.6f}")


Query: cost of attendance eastern illinois university
Passage i: Eastern Illinois University has roughly 8,000 students. Admission is selective. Tuition is approximately $8,550 per year...
Passage j: Lebanon, Illinois. Lebanon is a city in St. Clair County, Illinois, United States. The population was 5,523 at the 2010 ...
Stored g score (prob difference): 0.998756


In [10]:
test_point_i = compute_pointwise_ig_monot5(test_row["query"], test_row["passage_i"])
test_point_j = compute_pointwise_ig_monot5(test_row["query"], test_row["passage_j"])

test_prob_g = test_point_i["true_prob"] - test_point_j["true_prob"]
test_margin_g = test_point_i["margin"] - test_point_j["margin"]

print(f"Pointwise true prob i: {test_point_i['true_prob']:.6f}")
print(f"Pointwise true prob j: {test_point_j['true_prob']:.6f}")
print(f"Derived g from true probabilities: {test_prob_g:.6f}")
print(f"Derived g from margins: {test_margin_g:.6f}")
print(f"Convergence delta i: {test_point_i['convergence_delta']:.6f}")
print(f"Convergence delta j: {test_point_j['convergence_delta']:.6f}")

for label, result in [("doc_i", test_point_i), ("doc_j", test_point_j)]:
    print(f"\nTop {label} document words:")
    if len(result["doc"]["word_scores"]) == 0:
        print("  <no document words after truncation>")
        continue
    top_idx = np.argsort(result["doc"]["word_scores"])[::-1][:10]
    for idx in top_idx:
        print(f"  {result['doc']['word_tokens'][idx]:<20} {result['doc']['word_scores'][idx]:.4f}")


Pointwise true prob i: 0.998757
Pointwise true prob j: 0.000001
Derived g from true probabilities: 0.998756
Derived g from margins: 20.520908
Convergence delta i: -0.025646
Convergence delta j: -0.015517

Top doc_i document words:
  cost-of-attendance   0.1577
  $2,762.32.           0.0678
  $8,550               0.0529
  Illinois             0.0470
  $24,640              0.0416
  $10,680              0.0416
  Tuition              0.0343
  Illinois             0.0333
  Tuition              0.0293
  approximately        0.0289

Top doc_j document words:
  ▁a                   0.0866
  ▁a                   0.0269
  area.                0.0157
  St.                  0.0137
  region               0.0126
  the                  0.0099
  5,523                0.0093
  metropolitan         0.0075
  St.                  0.0067
  places               0.0064


In [11]:
attribution_records = []
failed_pairs = []
point_cache = {}

def cache_key(qid, pid):
    return (str(qid), str(pid))

for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc="Computing monoT5 pointwise IG"):
    try:
        key_i = cache_key(row["qid"], row["pid_i"])
        key_j = cache_key(row["qid"], row["pid_j"])

        if key_i not in point_cache:
            point_cache[key_i] = compute_pointwise_ig_monot5(row["query"], row["passage_i"])
        if key_j not in point_cache:
            point_cache[key_j] = compute_pointwise_ig_monot5(row["query"], row["passage_j"])

        pointwise_ig_i = point_cache[key_i]
        pointwise_ig_j = point_cache[key_j]

        attribution_records.append({
            "qid": row["qid"],
            "query": row["query"],
            "pid_i": row["pid_i"],
            "pid_j": row["pid_j"],
            "g_score": row["g_score"],
            "correct_pref": row["correct_pref"],
            "pointwise_ig_i": pointwise_ig_i,
            "pointwise_ig_j": pointwise_ig_j,
            "pointwise_trueprob_g": float(pointwise_ig_i["true_prob"] - pointwise_ig_j["true_prob"]),
            "pointwise_margin_g": float(pointwise_ig_i["margin"] - pointwise_ig_j["margin"]),
        })
    except Exception as e:
        failed_pairs.append({
            "qid": row["qid"],
            "pid_i": row["pid_i"],
            "pid_j": row["pid_j"],
            "error": str(e),
        })
        print(f"Error on qid={row['qid']}, pid_i={row['pid_i']}, pid_j={row['pid_j']}: {e}")

print(f"\nSuccessfully attributed: {len(attribution_records)} pairs")
print(f"Unique pointwise computations cached: {len(point_cache)}")
if failed_pairs:
    print(f"Failed: {len(failed_pairs)} pairs")


Python(8944) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Computing monoT5 pointwise IG:   0%|          | 0/90 [00:00<?, ?it/s]


Successfully attributed: 90 pairs
Unique pointwise computations cached: 108


In [12]:
prob_gaps = [record["pointwise_trueprob_g"] for record in attribution_records]
margin_gaps = [record["pointwise_margin_g"] for record in attribution_records]
deltas_i = [record["pointwise_ig_i"]["convergence_delta"] for record in attribution_records]
deltas_j = [record["pointwise_ig_j"]["convergence_delta"] for record in attribution_records]

print("Convergence delta - monoT5 pointwise IG:")
print(f" i mean={np.mean(deltas_i):.6f}, i max_abs={np.max(np.abs(deltas_i)):.6f}")
print(f" j mean={np.mean(deltas_j):.6f}, j max_abs={np.max(np.abs(deltas_j)):.6f}")

print("\nDerived pairwise g from true probabilities:")
print(f" mean={np.mean(prob_gaps):.6f}, min={np.min(prob_gaps):.6f}, max={np.max(prob_gaps):.6f}")

print("\nDerived pairwise g from margins:")
print(f" mean={np.mean(margin_gaps):.6f}, min={np.min(margin_gaps):.6f}, max={np.max(margin_gaps):.6f}")


Convergence delta - monoT5 pointwise IG:
 i mean=0.001511, i max_abs=0.213920
 j mean=-0.029837, j max_abs=1.461961

Derived pairwise g from true probabilities:
 mean=0.731395, min=-0.782165, max=0.999027

Derived pairwise g from margins:
 mean=12.134167, min=-4.985370, max=20.993263


In [13]:
def show_top_words(record, side="i", n=10):
    attr = record["pointwise_ig_i"] if side == "i" else record["pointwise_ig_j"]
    words = attr["doc"]["word_tokens"]
    scores = attr["doc"]["word_scores"]

    top_pos = np.argsort(scores)[::-1][:n]
    top_neg = np.argsort(scores)[:n]

    print(f"Side: doc_{side}")
    print(f"Query: {record['query']}")
    print(f"Stored g_score: {record['g_score']:.6f}")
    print(f"Derived true-prob g: {record['pointwise_trueprob_g']:.6f}")
    print(f"Direct true_prob: {attr['true_prob']:.6f}")

    print("\nTop positive document words:")
    for idx in top_pos:
        print(f"  {words[idx]:<20} {scores[idx]:.4f}")

    print("\nTop negative document words:")
    for idx in top_neg:
        print(f"  {words[idx]:<20} {scores[idx]:.4f}")

example = attribution_records[0]
show_top_words(example, side="i", n=10)
print("\n" + "=" * 80 + "\n")
show_top_words(example, side="j", n=10)


Side: doc_i
Query: cost of attendance eastern illinois university
Stored g_score: 0.998756
Derived true-prob g: 0.998756
Direct true_prob: 0.998757

Top positive document words:
  cost-of-attendance   0.1577
  $2,762.32.           0.0678
  $8,550               0.0529
  Illinois             0.0470
  $24,640              0.0416
  $10,680              0.0416
  Tuition              0.0343
  Illinois             0.0333
  Tuition              0.0293
  approximately        0.0289

Top negative document words:
  Eastern              -0.0451
  states,              -0.0279
  University           -0.0217
  year                 -0.0116
  Additional           -0.0105
  year.                -0.0099
  ▁8,000               -0.0092
  The                  -0.0087
  is                   -0.0079
  selective.           -0.0071


Side: doc_j
Query: cost of attendance eastern illinois university
Stored g_score: 0.998756
Derived true-prob g: 0.998756
Direct true_prob: 0.000001

Top positive document words:
  

In [14]:
out_path = out_dir / "attributions.pkl"
with open(out_path, "wb") as f:
    pickle.dump(attribution_records, f)

print(f"Saved {len(attribution_records)} attribution records -> {out_path}")

summary = pd.DataFrame([
    {
        "qid": record["qid"],
        "pid_i": record["pid_i"],
        "pid_j": record["pid_j"],
        "g_score": record["g_score"],
        "pointwise_trueprob_g": record["pointwise_trueprob_g"],
        "pointwise_margin_g": record["pointwise_margin_g"],
        "correct_pref": record["correct_pref"],
        "pi_true_prob": record["pointwise_ig_i"]["true_prob"],
        "pj_true_prob": record["pointwise_ig_j"]["true_prob"],
        "pi_margin": record["pointwise_ig_i"]["margin"],
        "pj_margin": record["pointwise_ig_j"]["margin"],
        "pi_ig_delta": record["pointwise_ig_i"]["convergence_delta"],
        "pj_ig_delta": record["pointwise_ig_j"]["convergence_delta"],
        "doc_i_top_word": record["pointwise_ig_i"]["doc"]["word_tokens"][np.argmax(record["pointwise_ig_i"]["doc"]["word_scores"])] if len(record["pointwise_ig_i"]["doc"]["word_tokens"]) > 0 else "",
        "doc_j_top_word": record["pointwise_ig_j"]["doc"]["word_tokens"][np.argmax(record["pointwise_ig_j"]["doc"]["word_scores"])] if len(record["pointwise_ig_j"]["doc"]["word_tokens"]) > 0 else "",
    }
    for record in attribution_records
])

summary_path = out_dir / "attributions_summary.csv"
summary.to_csv(summary_path, index=False)
print(f"Saved summary CSV -> {summary_path}")
summary.head()


Saved 90 attribution records -> ../outputs_monot5/attributions.pkl
Saved summary CSV -> ../outputs_monot5/attributions_summary.csv


,qid,pid_i,pid_j,g_score,pointwise_trueprob_g,pointwise_margin_g,correct_pref,pi_true_prob,pj_true_prob,pi_margin,pj_margin,pi_ig_delta,pj_ig_delta,doc_i_top_word,doc_j_top_word
0,1049774,7185662,2442388,0.998756,0.998756,20.520908,1,0.998757,9.834903e-07,6.688751,-13.832157,-0.025646,-0.015517,cost-of-attendance,▁a
1,1049774,7185662,4012312,0.998755,0.998755,19.718992,1,0.998757,2.192993e-06,6.688751,-13.030241,-0.025646,-0.172915,cost-of-attendance,▁a
2,1049774,7185662,2536875,0.998755,0.998755,20.201599,1,0.998757,1.353456e-06,6.688751,-13.512848,-0.025646,-0.023544,cost-of-attendance,Atmospheric
3,1049774,7185662,4872508,0.998714,0.998714,16.741339,1,0.998757,4.307230e-05,6.688751,-10.052588,-0.025646,-0.212575,cost-of-attendance,aid
4,1049774,7185662,7759529,0.998754,0.998754,19.670828,1,0.998757,2.301202e-06,6.688751,-12.982077,-0.025646,0.000888,cost-of-attendance,out-of-state
